<a href="https://colab.research.google.com/github/PARODBE/UAX_Fundamento_Ciencia_de_Datos/blob/main/Fase2_Lectura_y_Calidad_Auto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 2. Organización y entendimiento de los datos
Proyecto: Cuando la química compromete la cura


La etiqueta significa 0 = DILI negativa y 1 = DILI positiva. Fuente: página oficial de DILIst de la FDA.

In [3]:
# Instalo lo necesario para leer el Excel desde la web.
!pip -q install requests openpyxl

In [4]:
# Descargo el archivo original de la FDA y lo leo desde memoria.
import re, requests, pandas as pd
from io import BytesIO
from IPython.display import display

# Pruebo primero la FDA; si Colab recibe su página antiabuso, uso una copia pública del mismo Excel.
fuentes = [
    'https://www.fda.gov/media/160597/download',
    'https://raw.githubusercontent.com/tlee23-ic/GNN_DILI/main/DILIst%20%282020%29.xlsx'
]
original = None
errores_descarga = []
for fuente in fuentes:
    try:
        respuesta = requests.get(fuente, headers={'User-Agent':'Mozilla/5.0'}, timeout=60)
        respuesta.raise_for_status()
        # Un archivo xlsx empieza por PK; así no intentos leer la página de bloqueo.
        if not respuesta.content.startswith(b'PK'):
            raise ValueError('La respuesta no parece ser un archivo Excel.')
        original = pd.read_excel(BytesIO(respuesta.content))
        print('Archivo de DILIst descargado correctamente.')
        break
    except Exception as error:
        errores_descarga.append(str(error))
        print('No se pudo usar esta fuente; se prueba la siguiente.')
if original is None:
    raise RuntimeError('Fallaron ambas fuentes de DILIst: ' + ' | '.join(errores_descarga))
original.columns = original.columns.astype(str).str.strip()
print('Tamaño del Excel:', original.shape)
display(original.head())

# Busco las tres columnas necesarias aunque varíe la puntuación.
limpiar = lambda x: re.sub(r'[^a-z0-9]', '', str(x).lower())
cols = {limpiar(c): c for c in original.columns}
id_col = cols.get('dilistid')
nombre_col = cols.get('compoundname') or cols.get('drugname')
clase_col = cols.get('dilistclassification') or cols.get('classification')
if not all([id_col, nombre_col, clase_col]):
    raise ValueError('No se encontraron las columnas principales: ' + str(list(original.columns)))

df = original[[id_col, nombre_col, clase_col]].copy()
df.columns = ['DILIST_ID', 'CompoundName', 'DILIst_Classification']
df['DILIST_ID'] = pd.to_numeric(df['DILIST_ID'], errors='coerce')
df['DILIst_Classification'] = pd.to_numeric(df['DILIst_Classification'], errors='coerce')
df['CompoundName'] = df['CompoundName'].astype('string').str.strip()
display(df.head())
display(df.dtypes.to_frame('tipo'))

Archivo de DILIst descargado correctamente.
Tamaño del Excel: (1279, 4)


,DILIST_ID,CompoundName,DILIst Classification,Routs of Administration
0,1,mercaptopurine,1,Oral
1,2,acetaminophen,1,Oral
2,3,azathioprine,1,Oral
3,4,chlorpheniramine,0,Oral
4,5,clofibrate,1,Oral


,DILIST_ID,CompoundName,DILIst_Classification
0,1,mercaptopurine,1
1,2,acetaminophen,1
2,3,azathioprine,1
3,4,chlorpheniramine,0
4,5,clofibrate,1


,tipo
DILIST_ID,int64
CompoundName,string[python]
DILIst_Classification,int64


In [5]:
# Reviso faltantes, duplicados y valores distintos.
calidad = pd.DataFrame({
    'faltantes': df.isna().sum(),
    'valores_distintos': df.nunique()
})
display(calidad)
print('Filas duplicadas:', df.duplicated().sum())
print('IDs repetidos:', df['DILIST_ID'].duplicated().sum())
print('Nombres repetidos:', df['CompoundName'].str.casefold().duplicated().sum())

# Cuento las clases DILI.
clases = df['DILIst_Classification'].value_counts().sort_index().to_frame('compuestos')
clases['porcentaje'] = (100 * clases['compuestos'] / len(df)).round(1)
display(clases.rename(index={0:'DILI negativa', 1:'DILI positiva'}))

,faltantes,valores_distintos
DILIST_ID,0,1279
CompoundName,0,1279
DILIst_Classification,0,2


Filas duplicadas: 0
IDs repetidos: 0
Nombres repetidos: 0


,compuestos,porcentaje
DILIst_Classification,,
DILI negativa,511,40.0
DILI positiva,768,60.0


Hay un 60% de compuestos que son hepatotóxicos y un 40% que no.

In [6]:
# Guardo el CSV en el espacio temporal de Colab.
df.to_csv('dilist_trabajo.csv', index=False)
calidad.to_csv('resumen_calidad_fase2.csv')
clases.to_csv('clases_fase2.csv')
print('CSV y resúmenes creados. No se requiere descargarlos para ejecutar el análisis.')

CSV y resúmenes creados. No se requiere descargarlos para ejecutar el análisis.
